# Turniting AI Detector — GPU training on Colab

Runs the full transformer recipe (`configs/training_gpu.yaml`: mDeBERTa-v3-base, 512 tokens, stride 128, all layers, 150k windows, 3 epochs, AMP) on the same `TURNITING_DATASET_V1` splits and frozen evaluation sets as the CPU run, so results are directly comparable.

**Before you start**
1. On your PC: `python scripts/export_for_colab.py` and upload `datasets/cache/turniting_colab_bundle.zip` to `MyDrive/turniting/`.
2. Runtime → Change runtime type → **GPU** (T4 works; L4/A100 are faster).
3. If the GitHub repo is private, add a Colab secret `GITHUB_TOKEN` (read-only PAT) in the 🔑 panel.

Checkpoints, models and reports live in Drive, so if Colab disconnects just **re-run all cells**: training resumes from `last`.

Independent detector — not a Turnitin model. No Turnitin reports are used here.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/turniting'
import os; os.makedirs(DRIVE, exist_ok=True)

In [ ]:
import os, subprocess
from google.colab import userdata
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
url = f'https://{token}@github.com/Anthgg/Turniting.git' if token else 'https://github.com/Anthgg/Turniting.git'
if not os.path.exists('/content/Turniting'):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'exp/ai-detector-training', url, '/content/Turniting'], check=True)
else:
    subprocess.run(['git', '-C', '/content/Turniting', 'pull', '--ff-only'], check=True)
%cd /content/Turniting/training

In [ ]:
# torch/numpy/pandas/sklearn/pyarrow come with Colab; pin the libraries the code depends on.
!pip -q install transformers==4.46.3 accelerate==1.1.1 sentencepiece==0.2.0 langid==1.1.6 datasketch==1.6.5 tiktoken protobuf

In [ ]:
# Data + persistent state on Drive (survives disconnects)
import zipfile, os
W = f'{DRIVE}/workdir'
if not os.path.exists(f'{W}/datasets/normalized/master'):
    with zipfile.ZipFile(f'{DRIVE}/turniting_colab_bundle.zip') as z:
        z.extractall(W)
for sub in ['checkpoints', 'models', 'reports/eval']:
    os.makedirs(f'{W}/{sub}', exist_ok=True)
# point the repo's training/ folders at Drive
!rm -rf datasets/normalized datasets/manifests checkpoints models
!mkdir -p datasets && ln -s {W}/datasets/normalized datasets/normalized && ln -s {W}/datasets/manifests datasets/manifests
!ln -s {W}/checkpoints checkpoints && ln -s {W}/models models
!ls datasets/normalized/master models

In [ ]:
import json, os, time, glob
ids_file = f'{W}/experiment_ids.json'
if os.path.exists(ids_file):
    ids = json.load(open(ids_file))
else:
    stamp = time.strftime('%Y%m%d_%H%M')
    ids = {'global': f'{stamp}_mdeberta_gpu_v1', 'refine': f'{stamp}_mdeberta_gpu_es_acad_v1'}
    json.dump(ids, open(ids_file, 'w'))
ids

def train(mode, exp_id, extra=()):
    exp = f'checkpoints/{exp_id}'
    if os.path.exists(f'{exp}/experiment.json'):
        print(exp_id, 'already finished'); return
    args = ['python', 'scripts/train_transformer.py', '--config', 'configs/training_gpu.yaml', '--mode', mode]
    if os.path.exists(f'{exp}/last/trainer_state.pt'):
        args += ['--resume-from-checkpoint', f'{exp}/last']   # resume after a disconnect
    else:
        args += ['--experiment-id', exp_id, *extra]
    print(' '.join(args)); !{' '.join(args)}

In [ ]:
# GLOBAL_MULTILINGUAL_TRANSFORMER (early stopping on validation macro F1, patience 2)
train('global', ids['global'])

In [ ]:
# SPANISH_ACADEMIC_TRANSFORMER: continues from the global best, 70 % Spanish/academic + 30 % replay, lower LR
train('refine', ids['refine'], ['--init-from', f"checkpoints/{ids['global']}/best"])

In [ ]:
g, r = ids['global'], ids['refine']
!python scripts/calibrate.py --experiment checkpoints/{g} --out models/gpu/global --name GLOBAL_MULTILINGUAL_TRANSFORMER_GPU --link-weights
!python scripts/calibrate.py --experiment checkpoints/{r} --out models/gpu/spanish_academic --name SPANISH_ACADEMIC_TRANSFORMER_GPU --link-weights
!python scripts/calibrate.py --experiment checkpoints/{r} --out models/gpu/calibrated_spanish_academic --name CALIBRATED_SPANISH_ACADEMIC_TRANSFORMER_GPU --calibrate --link-weights

In [ ]:
# 1) VALIDATION only -> choose (compare with the CPU run's reports/model_selection.json)
for name, path in [('GLOBAL_MULTILINGUAL_TRANSFORMER_GPU', 'models/gpu/global'),
                   ('SPANISH_ACADEMIC_TRANSFORMER_GPU', 'models/gpu/spanish_academic'),
                   ('CALIBRATED_SPANISH_ACADEMIC_TRANSFORMER_GPU', 'models/gpu/calibrated_spanish_academic')]:
    !python scripts/evaluate.py --model {path} --name {name}__validation

In [ ]:
# 2) TEST-side sets, once, after the choice above
for name, path in [('GLOBAL_MULTILINGUAL_TRANSFORMER_GPU', 'models/gpu/global'),
                   ('SPANISH_ACADEMIC_TRANSFORMER_GPU', 'models/gpu/spanish_academic'),
                   ('CALIBRATED_SPANISH_ACADEMIC_TRANSFORMER_GPU', 'models/gpu/calibrated_spanish_academic')]:
    !python scripts/evaluate.py --model {path} --name {name} --final

In [ ]:
# Copy reports to Drive (weights already live there under workdir/checkpoints)
!cp -r reports/eval/* {W}/reports/eval/ && cp checkpoints/*/experiment.json {W}/reports/ 2>/dev/null; ls {W}/reports/eval